In [ ]:
# Cell 1: Install dependencies
!pip install -q --upgrade torch==2.5.1 transformers==4.44.0 datasets==2.21.0 peft==0.13.0 trl==0.9.6 accelerate==0.33.0 bitsandbytes==0.46.1 huggingface_hub==0.24.0

In [ ]:
# Cell 2: Check GPU and setup
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    torch.cuda.empty_cache()

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Cell 3: Clone repo and prepare data
import os
os.chdir('/content')
!rm -rf myai
!git clone https://github.com/kambleaa007/my.ai.llm.git myai
os.chdir('/content/myai/finetuning')

# Create training data
import json
data = [
    {"instruction": "मराठीत भाषांतर करा: Hello, how are you?", "output": "नमस्कार, तुमची कशी आहे?"},
    {"instruction": "मराठीत भाषांतर करा: What is your name?", "output": "तुमचे नाव काय आहे?"},
    {"instruction": "मराठीत भाषांतर करा: I am learning Marathi", "output": "मी मराठी शिकत आहे."},
    {"instruction": "मराठीत भाषांतर करा: Thank you very much", "output": "खूप खूप धन्यवाद."},
    {"instruction": "मराठीत भाषांतर करा: Good morning", "output": "सुप्रभात."},
    {"instruction": "मराठीत भाषांतर करा: Good night", "output": "शुभ रात्री."},
    {"instruction": "मराठीत भाषांतर करा: How was your day?", "output": "तुमचा दिवस कसा गेला?"},
    {"instruction": "मराठीत भाषांतर करा: I love Maharashtra", "output": "मला महाराष्ट्र आवडते."},
    {"instruction": "मराठी व्याकरण स्पष्ट करा: सर्वनाम", "output": "सर्वनाम हे शब्द आहे ज्याचे वापर नावाचे ठिकाणी केले जाते. उदा: तो, ती, ते, मी, आम्ही, तुम्ही."},
    {"instruction": "मराठी व्याकरण स्पष्ट करा: क्रियापद", "output": "क्रियापद हे शब्द आहे ज्यामुळे कृती किंवा अवस्था दर्शवते. उदा: चालणे, खाणे, बोलणे, असणे."},
    {"instruction": "मराठी व्याकरण स्पष्ट करा: विशेषण", "output": "विशेषण हे शब्द आहे ज्यामुळे नावाचे गुण धर्म दर्शवते. उदा: मोठा, लहान, सुंदर, चांगला."},
    {"instruction": "मराठी म्हणी स्पष्ट करा: 'घरचा भेदी लंका ढहला'", "output": "हे म्हणी म्हणजे आतचं भेद घेऊन कोणी बाहेरील व्यक्तीला सांगल्यास तो हानी पोहोचवतो."},
    {"instruction": "मराठी म्हणी स्पष्ट करा: 'एक हाताने ताली वाजत नाही'", "output": "हे म्हणी म्हणजे कोणतेही कार्य एकच व्यक्तीने करता येत नाही, बरं कार्यासाठी अनेकांची मदत हवी असते."},
    {"instruction": "मराठी कविता लिहा: पावसाबरोबरच", "output": "पावस आला, धरा झाली नम,\nवारा वावत, मन झालं चंचल.\nपान झरे, नदीची धार वाढली,\nपक्षी बोलत, निसर्ग झालं सुंदर."},
    {"instruction": "मराठी कविता लिहा: आईवर", "output": "आई माझी, जगात सर्वात महान,\nमाझ्या साठी तीच देते सर्व बलिदान.\nतिचे हात, माझ्या डोळात आणते सुख,\nआईचं प्रेम, आहे जगात अनमोल सत्य."},
]

with open("marathi_instruct.jsonl", "w", encoding="utf-8") as f:
    for item in data:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Created {len(data)} training samples")

In [ ]:
# Cell 4: Load model with 4-bit quantization
import torch
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

gc.collect()
torch.cuda.empty_cache()

model_id = "google/gemma-2-9b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Cell 5: Prepare dataset
from datasets import load_dataset

dataset = load_dataset("json", data_files="marathi_instruct.jsonl", split="train")

def format_chat(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}

dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
print(f"Dataset size: {len(dataset)}")
print(dataset[0])

In [ ]:
# Cell 6: Training
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="marathi-gemma-2-9b-qlora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    warmup_steps=50,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    tokenizer=tokenizer,
    max_seq_length=2048,
    packing=False,
)

trainer.train()

In [ ]:
# Cell 7: Save model
trainer.save_model("marathi-gemma-2-9b-qlora-final")
tokenizer.save_pretrained("marathi-gemma-2-9b-qlora-final")
print("Model saved!")

import torch
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 8: Test inference
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-9b-it",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base_model, "marathi-gemma-2-9b-qlora-final")
model.eval()

def generate(prompt):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=200,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][input_ids.shape[1]:], skip_special_tokens=True)

print(generate("मराठीत भाषांतर करा: Hello, how are you?"))
print(generate("मराठी व्याकरण स्पष्ट करा: विशेषण"))
print(generate("मराठी कविता लिहा: प्रकृतीवर"))

In [ ]:
# Cell 9: Convert to GGUF (optional - requires llama.cpp)
# This step requires llama.cpp which may not work well on Colab T4
# Better to download the model and convert locally

# If you want to push to Hugging Face Hub:
# from huggingface_hub import HfApi
# HfApi().upload_folder(
#     folder_path="marathi-gemma-2-9b-qlora-final",
#     repo_id="kambleaa007/marathi-gemma-2-9b-qlora",
#     repo_type="model"
# )